# OpenRouter test call

This notebook sends a simple prompt to OpenRouter using the DeepSeek V4 Flash (free) model.

Set your API token in the environment before running:


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import json
import requests
import os

load_dotenv(Path("../.env.secret"))

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError("OPENROUTER_API_KEY is not set")

prompt = """
    You are a helpful recipe recommendation assistant.

    A user will provide:
    1. A list of ingredients they currently have at home
    2. Any dietary restrictions or allergies they have

    You will be given one or more recipes in the following JSON format:
    {
    "title": "...",
    "ingredients": [...],
    "ner": [...],         // simplified ingredient names
    "directions": [...],
    "link": "...",
    "source": "..."
    }

    Your task:
    - Check each recipe against the user's available ingredients and restrictions
    - For each recipe that is a good match, explain WHY it suits the user:
    - Which of their ingredients are used
    - What (if anything) they might be missing and how easy it is to substitute or skip
    - Why it is safe given their allergies/restrictions
    - Why it would be a good choice overall (taste, simplicity, nutrition, etc.)
    - If a recipe is NOT suitable (e.g. contains an allergen), clearly say so and briefly explain why

    ---

    User input:
    Ingredients I have: milk, sugar, vanilla, butter
    Dietary restrictions / allergies: chili

    Recipes to evaluate:
    {
    "title":"No-Bake Nut Cookies"
    "ingredients":[
    0:"1 c. firmly packed brown sugar"
    1:"1/2 c. evaporated milk"
    2:"1/2 tsp. vanilla"
    3:"1/2 c. broken nuts (pecans)"
    4:"2 Tbsp. butter or margarine"
    5:"3 1/2 c. bite size shredded rice biscuits"
    ]
    "parsed_ingredients":[

    ]
    "directions":[
    0:"In a heavy 2-quart saucepan, mix brown sugar, nuts…"
    1:"Stir over medium heat until mixture bubbles all ov…"
    2:"Boil and stir 5 minutes more. Take off heat."
    3:"Stir in vanilla and cereal; mix well."
    4:"Using 2 teaspoons, drop and shape into 30 clusters…"
    5:"Let stand until firm, about 30 minutes."
    ]
    "link":"www.cookbooks.com/Recipe-Details.aspx?id=44874"
    "source":"Gathered"
    "ner":[
    0:"brown sugar"
    1:"milk"
    2:"vanilla"
    3:"nuts"
    4:"butter"
    5:"bite size shredded rice biscuits"
    ]
    }

    Only use bullet points and keep the reasoning concise. Wrap all the bullet points under: <positive></positive> and <negative></negative> tags to indicate suitability.
"""

url = "https://openrouter.ai/api/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
}

payload = {
    "model": "openrouter/free",
    "messages": [{"role": "user", "content": prompt}],
}

response = requests.post(url, headers=headers, data=json.dumps(payload), timeout=30)
response.raise_for_status()

result = response.json()
print(result["choices"][0]["message"]["content"].strip())

<positive>
- **Ingredient matches**: Uses your milk, vanilla, and butter directly
- **Safety**: No chili or spicy ingredients - completely safe for your allergy
- **Easy substitutions**: 
  - Regular sugar can replace brown sugar (or make quick brown sugar with sugar + molasses)
  - Regular milk works instead of evaporated milk (texture will be slightly different but still tasty)
  - Rice biscuits could substitute with any crispy cereal you have
- **Good choice**: Simple no-bake treat, uses pantry staples, and nuts add protein/fat for satisfaction
</positive>

<negative>
- **Missing items**: You'll need nuts (can omit for nut-free version) and possibly brown sugar/rice biscuits
- **Note**: Recipe calls for "broken nuts" but you can easily skip or use alternative crunchy ingredients
</negative>
</assistant>
